**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/asari-2009"], check=True)
    os.chdir("/content/encore/entries/asari-2009")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of asari-2009-01, checked on CRCNS ac-1

Paper. Asari H, Zador AM (2009). Long-lasting context dependence constrains neural encoding
models in rodent auditory cortex. J Neurophysiol 102:2638-2656.
[doi:10.1152/jn.00577.2009](https://doi.org/10.1152/jn.00577.2009)

Dataset. CRCNS ac-1, [doi:10.6080/K0KW5CXR](https://doi.org/10.6080/K0KW5CXR), streamed from the
public S3 bucket `crcnsarchive` (`ac-1/`), zip files opened in place. CRCNS data sets are not
versioned, so the version is recorded as `unversioned` (docs/archives.md, CRCNS); the MD5 sums of
the zip files are in `asari.MD5`.

This notebook is the reconcile step of attempt asari-2009-02 (docs/protocol.md, "Repeat
attempts"). It checks, on this version, the findings of the earlier attempt asari-2009-01 that
asari-2009-02 did not report itself: findings 5, 6 and 10. Each section opens with the earlier
statement. A last section tests the likeliest reason for two outcomes that differ between the
attempts. It uses the loading and measures of `asari.py`, the helper of
`reproduce_asari_2009.ipynb`.

In [ ]:
import time
import numpy as np
import pandas as pd

import asari

pd.set_option("display.width", 200)
T0 = time.time()
A = asari.Archive()
rows = pd.DataFrame(asari.summary(A))
print(f"data set {asari.DATASET_ID} ({asari.DATASET_DOI}), version {asari.DATASET_VERSION}; zip members: {len(A.members)}")
print("Summary.mat rows by experiment:", rows.groupby("experiment").size().to_dict())

## Earlier finding 5

> Summary.mat lists two cells, 20060314-ha1-001 and 20060320-ha3-003, in both the natural sound
> and the synthetic sound group, so their tuning-curve files are named twice. The recordings are
> the same files; no two files hold identical traces.

Every cell of Summary.mat listed in more than one experiment group, with the files listed under each.

In [ ]:
groups = rows.groupby("cell").experiment.agg(lambda s: sorted(set(s)))
multi = groups[groups.map(len) > 1]
print("cells in more than one group:", multi.to_dict())
for c in multi.index:
    print(rows[rows.cell == c][["experiment", "file", "description"]].to_string(index=False))
dup = rows[rows.duplicated("file", keep=False)].sort_values("file")
print("\nfiles listed more than once:")
print(dup[["experiment", "cell", "file", "description"]].to_string(index=False))

Result: still present. 20060314-ha1-001 and 20060320-ha3-003 are listed in the natural-sound
group (0) and the synthetic-sound group (2), and their tuning-curve file (-001) is listed in both.
Their sequence recordings differ between the groups (natural: -004 and -003; synthetic: -002 and
-003, and -002).

## Earlier finding 6

> Question for the authors. Probes last 2 to 6 s. How do the population curves of Figures 3, 4, 8
> and 9D treat times after a probe has ended: is the average at time t over all probes, an ended
> probe counting as zero (or not significant), or over the probes still running? The late fraction
> of Figure 3A matches the paper (0.252 against 0.237) only with the first reading.

The natural-sound probes are formed and tested as in `reproduce_asari_2009.ipynb` (silence counted
as a context, the same seeds), and the late fraction of Figure 3A is computed with both readings
of the population average: ended probes counted as not significant (`all`), or the average over
the probes still running (`running`).

In [ ]:
nat_rows = rows[(rows.experiment == 0) & (rows.description != "tuning curve")].reset_index(drop=True)
SEED_NAT = 10_000   # as in reproduce_asari_2009.ipynb
t = time.time()
probes = []
for i, r in nat_rows.iterrows():
    _, res = asari.analyze_cc_cell(A, A.result_member(r.file), seed=SEED_NAT + i, extras=False)
    probes += [p for p in res if p["eligible"]]
print(f"{len(probes)} probes in {len(nat_rows)} files, {time.time() - t:.0f} s")
dur = np.array([p["duration"] for p in probes])
print("probe durations (s), min / median / max:", np.round([dur.min(), np.median(dur), dur.max()], 2))
print("probes still running at t:", {f"{s} s": int((dur > s).sum()) for s in range(1, 7)})
out = []
for reading in ["all", "running"]:
    prop = asari.population_mean(asari.stack(probes, "signif"), reading)
    noise = asari.population_mean(asari.stack(probes, "noise"), reading)
    lf, tpk, _ = asari.late_fraction(prop, noise)
    out.append(dict(reading=reading, late_fraction=round(lf, 3), cum_peak_s=tpk,
                    proportion_at_4s=round(float(prop[4000]), 4), proportion_at_5s=round(float(prop[5000]), 4)))
print(pd.DataFrame(out).to_string(index=False))
print("paper: late fraction 0.237")

Result: still present. The paper does not say how probes of different lengths (1.98 to 6.06 s)
are averaged once some have ended; 304 of the 305 probes still run at 2 s, 214 at 4 s and 24 at
6 s. The late fraction is 0.225 with ended probes counted as not significant and 0.237 over the
probes still running (paper 0.237). On this data both readings are within 10% of the paper's
value, so the earlier statement that only the first reading matches does not hold with this
attempt's probes; the two readings still differ.

## Earlier finding 10

> The paper's analysis script CxtDep.m centres each trial on its own mean over the probe period
> before computing the fractional powers, where the Methods centre the whole trace over time. It
> also counts silence as a context of every probe, which the Methods do not say. The attempt
> follows the script.

The Methods (Analysis, paragraph 1): "we applied a median filter (10-ms window) to clip spikes from
the raw data and centered the subthreshold responses to have zero mean [i.e., r(t) - <r(t)>_t,
instead of subtracting the resting potential; <.>_t indicates the average over time t]". The
lines of the authors' script in the data set that set the context of a probe and centre the
trials:

In [ ]:
src = A.read("asari/Tools/CxtDep.m").decode("latin-1").splitlines()
for a, b in [(105, 113), (139, 141), (340, 347)]:
    print("\n".join(f"{k + 1:4d} {src[k]}" for k in range(a - 1, b)))
    print("...")
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

Result: still present. `FracPow` in CxtDep.m (line 346) subtracts from each trial its own mean
over the probe window, and a whole-trace centring (line 347) is commented out; the Methods centre
the response over time without saying over which window. With no `param.blank` given, the script
sets the blank periods to 2.5 s (probe) and 4 s (context) (line 112) and includes the preceding
silence as a context of every probe (lines 139 to 141), which the Methods do not say.

## Outcomes that differ: the swapped fragment lists

asari-2009-02 cuts the probes of 20051108-ha1-001 with the fragment lists that fit each file's
triggers (its finding 2); asari-2009-01 cut them with the stored lists. The natural-sound probes of
the section on earlier finding 6 are recomputed with the stored lists for that cell only (same
seeds), for the measures of fig3a-proportion-significant and the natural-sound area under Q(t)
from 0 to 4 s that fig8c-relative-effects divides by.

In [ ]:
def q_area(ps):
    q = asari.q_curves(ps, "running")
    return float(np.nansum(q["Q"][q["t"] < 4.0]) * 1e-3)

CELL = "20051108-ha1-001"
stored = [p for p in probes if CELL not in p["member"]]
for i, r in nat_rows[nat_rows.cell == CELL].iterrows():
    _, res = asari.analyze_cc_cell(A, A.result_member(r.file), seed=SEED_NAT + i, extras=False, fix_swaps=False)
    stored += [p for p in res if p["eligible"]]
out = []
for name, ps in [("lists that fit the triggers", probes), ("stored lists", stored)]:
    row = dict(fragment_lists=name, probes=len(ps), significant=sum(p["significant"] for p in ps),
               significant_in_cell=sum(p["significant"] for p in ps if CELL in p["member"]))
    for reading in ["all", "running"]:
        prop = asari.population_mean(asari.stack(ps, "signif"), reading)
        noise = asari.population_mean(asari.stack(ps, "noise"), reading)
        lf, tpk, _ = asari.late_fraction(prop, noise)
        row[f"late_fraction_{reading}"] = round(lf, 3)
        row[f"cum_peak_s_{reading}"] = tpk
    row["natural_area_0_4s"] = round(q_area(ps), 3)
    out.append(row)
print(pd.DataFrame(out).to_string(index=False))
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

Result. With the stored lists, 186 of 305 probes are significant, the late fraction is 0.252 over
all probes and 0.267 over the running ones, with the cumulative curve peaking at 4.78 s: the
values asari-2009-01 recorded. With the lists that fit the triggers they are 183, 0.225 and 0.237,
peaking at 4.718 s, as asari-2009-02 recorded. The natural-sound area under Q(t) from 0 to 4 s is
0.762 s with the stored lists (asari-2009-01 gave 0.752 s) and 0.525 s with the corrected ones.